In [6]:
import collections
import sys

def get_mst(vertices, edges):
    parent = {v: v for v in vertices}
    rank = {v: 0 for v in vertices}
    
    def find(item):
        if parent[item] == item:
            return item
        parent[item] = find(parent[item])
        return parent[item]
        
    def union(v1, v2):
        root1 = find(v1)
        root2 = find(v2)
        if root1 != root2:
            if rank[root1] > rank[root2]:
                parent[root2] = root1
            else:
                parent[root1] = root2
                if rank[root1] == rank[root2]:
                    rank[root2] += 1
            return True
        return False

    sorted_edges = sorted(edges, key=lambda x: x[2])
    mst_edges = []
    mst_cost = 0
    for u, v, w in sorted_edges:
        if find(u) != find(v):
            union(u, v)
            mst_edges.append((u, v, w))
            mst_cost += w
    return mst_edges, mst_cost

def dijkstra(graph, start, end):
    distances = {vertex: float('infinity') for vertex in graph}
    distances[start] = 0
    previous_vertices = {vertex: None for vertex in graph}
    vertices = list(graph.keys())
    
    while vertices:
        current_vertex = min(vertices, key=lambda vertex: distances[vertex])
        vertices.remove(current_vertex)
        if distances[current_vertex] == float('infinity'):
            break
        for neighbor, weight in graph[current_vertex].items():
            alternative_route = distances[current_vertex] + weight
            if alternative_route < distances[neighbor]:
                distances[neighbor] = alternative_route
                previous_vertices[neighbor] = current_vertex
                
    path, current_vertex = collections.deque(), end
    while previous_vertices[current_vertex] is not None:
        path.appendleft(current_vertex)
        current_vertex = previous_vertices[current_vertex]
    if path:
        path.appendleft(current_vertex)
    return list(path), distances[end]

def bfs(graph, start):
    visited = set()
    queue = collections.deque([start])
    order = []
    visited.add(start)
    while queue:
        vertex = queue.popleft()
        order.append(vertex)
        for neighbor in sorted(graph[vertex].keys()):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

def dfs(graph, start):
    visited = set()
    order = []
    def dfs_helper(vertex):
        visited.add(vertex)
        order.append(vertex)
        for neighbor in sorted(graph[vertex].keys()):
            if neighbor not in visited:
                dfs_helper(neighbor)
    dfs_helper(start)
    return order

def main():
    choice = input("Do you want to enter custom data? (yes/no): ").strip().lower()
    
    if choice == 'yes':
        print("\n--- Enter Graph Infrastructure Data ---")
        vertices_input = input("Enter all city/node names separated by spaces: ").strip().split()
        vertices = [v for v in vertices_input if v]
        
        graph = {v: {} for v in vertices}
        congested_graph = {v: {} for v in vertices}
        edges_list = []
        
        print("\nEnter connections. Type 'done' when finished.")
        while True:
            edge_str = input("Enter connection (Format: Node1 Node2 standard_cost congested_cost) or 'done': ").strip()
            if edge_str.lower() == 'done':
                break
            parts = edge_str.split()
            if len(parts) != 4:
                print("Invalid format. Please try again.")
                continue
            u, v, cost_str, cong_cost_str = parts
            if u not in graph or v not in graph:
                print("One or both nodes not found in your nodes list. Try again.")
                continue
            try:
                cost = float(cost_str)
                cong_cost = float(cong_cost_str)
            except ValueError:
                print("Costs must be numbers. Try again.")
                continue
                
            graph[u][v] = cost
            graph[v][u] = cost
            congested_graph[u][v] = cong_cost
            congested_graph[v][u] = cong_cost
            edges_list.append((u, v, cost))
            
        source = input("\nEnter starting node for Route Planning & Traversals: ").strip()
        destination = input("Enter destination node: ").strip()
        if source not in graph or destination not in graph:
            print("Error: Invalid source or destination nodes. Exiting.")
            sys.exit()
    else:
        print("\n--- Running with built-in Network Example ---")
        vertices = ["A", "B", "C", "D", "E"]
        graph = {
            "A": {"B": 4, "C": 2},
            "B": {"A": 4, "C": 1, "D": 5},
            "C": {"A": 2, "B": 1, "D": 8, "E": 10},
            "D": {"B": 5, "C": 8, "E": 2},
            "E": {"C": 10, "D": 2}
        }
        congested_graph = {
            "A": {"B": 12, "C": 2},
            "B": {"A": 12, "C": 1, "D": 5},
            "C": {"A": 2, "B": 1, "D": 8, "E": 10},
            "D": {"B": 5, "C": 8, "E": 2},
            "E": {"C": 10, "D": 2}
        }
        edges_list = [
            ("A", "B", 4), ("A", "C", 2), ("B", "C", 1),
            ("B", "D", 5), ("C", "D", 8), ("C", "E", 10), ("D", "E", 2)
        ]
        source = "A"
        destination = "E"

    orig_path, orig_cost = dijkstra(graph, source, destination)
    updated_path, updated_cost = dijkstra(congested_graph, source, destination)
    bfs_order = bfs(graph, source)
    dfs_order = dfs(graph, source)
    mst_edges, mst_cost = get_mst(vertices, edges_list)

    print("\n" + "="*40)
    print("      VISUALIZATION AND REPORTING       ")
    print("="*40)
    
    print("\n[ROUTE PLANNING]")
    print(f"• Source and Destination: {source} -> {destination}")
    print(f"• Shortest Route Obtained: {' -> '.join(orig_path)}")
    print(f"• Total Travel Cost: {orig_cost}")
    
    print("\n[TRAFFIC ANALYSIS]")
    print(f"• Original Route: {' -> '.join(orig_path)} (Cost: {orig_cost})")
    print(f"• Updated Route after Congestion: {' -> '.join(updated_path)}")
    print(f"• Cost Comparison: Original Cost = {orig_cost} | New Post-Congestion Cost = {updated_cost}")
    
    print("\n[NETWORK CONNECTIVITY]")
    print(f"• BFS Traversal Order (starting from {source}): {' -> '.join(bfs_order)}")
    print(f"• DFS Traversal Order (starting from {source}): {' -> '.join(dfs_order)}")
    
    print("\n[INFRASTRUCTURE PLANNING]")
    print("• Roads Selected in the MST:")
    for u, v, w in mst_edges:
        print(f"  - {u} -- {v} (Cost: {w})")
    print(f"• Total Infrastructure Cost: {mst_cost}")
    print("="*40)

if __name__ == "__main__":
    main()

Do you want to enter custom data? (yes/no):  no



--- Running with built-in Network Example ---

      VISUALIZATION AND REPORTING       

[ROUTE PLANNING]
• Source and Destination: A -> E
• Shortest Route Obtained: A -> C -> B -> D -> E
• Total Travel Cost: 10

[TRAFFIC ANALYSIS]
• Original Route: A -> C -> B -> D -> E (Cost: 10)
• Updated Route after Congestion: A -> C -> B -> D -> E
• Cost Comparison: Original Cost = 10 | New Post-Congestion Cost = 10

[NETWORK CONNECTIVITY]
• BFS Traversal Order (starting from A): A -> B -> C -> D -> E
• DFS Traversal Order (starting from A): A -> B -> C -> D -> E

[INFRASTRUCTURE PLANNING]
• Roads Selected in the MST:
  - B -- C (Cost: 1)
  - A -- C (Cost: 2)
  - D -- E (Cost: 2)
  - B -- D (Cost: 5)
• Total Infrastructure Cost: 10
